# Quota-exhausted A3 offset neutralization demo

This notebook demonstrates the current upper-agent settle-window mechanism:

- apply one directional upper action: `b[g1->g2,eMBB] = -1.0`
- safe admission freezes a release quota for source `g1/eMBB`
- the negative A3 offset stays active while quota remains
- once quota becomes `0`, the matching negative offset is neutralized to `0 dB`

The reward measurement window should open only after this settle/calibration behavior.

In [1]:
from pathlib import Path
import sys

# Make this notebook import repo modules whether Jupyter starts in the repo
# root or inside notebooks/.
project_root = Path.cwd()
if not (project_root / "global_ppo_3gnb_env.py").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

from global_ppo_3gnb_env import GlobalPPO3GNBEnv
from upper_agent_training_scenarios import CENTER_GAP_GNB_CONFIGS

In [2]:
env = GlobalPPO3GNBEnv(
    seed=7,
    scenario_mode="curriculum",
    training_scenarios="high_load_inner_asymmetric",
    scenario_selection="cycle",
    gnb_configs=CENTER_GAP_GNB_CONFIGS["medium_270m"],
    upper_window_seconds=1.0,
    local_steps_per_global=12,
    radio_substeps=20,
    post_handover_settle_steps=9,
    max_handovers_per_local_step=1,
    safe_admission_enabled=True,
)

try:
    _obs, reset_info = env.reset(seed=7)

    action = np.zeros(env.action_space.shape, dtype=np.float32)
    directional = action.reshape(3, 2, 3)
    source_gnb = 1
    target_slot = 1
    target_gnb = env.neighbors[source_gnb][target_slot]
    slice_idx = 0
    slice_type = env.slice_types[slice_idx]
    directional[source_gnb, target_slot, slice_idx] = -1.0

    def useful_gnb_totals():
        matrix = env._current_useful_load_matrix()
        return matrix, np.sum(matrix, axis=1)

    def load_fields(prefix, totals):
        return {f"g{idx}_{prefix}": round(float(value), 3) for idx, value in enumerate(totals)}

    directional_bias = env._action_to_directional_bias_tensor(action)
    offsets, _offset_debug = env._compute_strong_local_offsets(directional_bias)
    env._apply_slice_offsets(offsets)
    env.base_env.begin_safe_admission_window(directional_bias, env.slice_types)
    env.base_env.begin_sla_window()

    live_offsets = env._zero_quota_exhausted_offsets(offsets)
    if not np.array_equal(live_offsets, offsets):
        env._apply_slice_offsets(live_offsets)

    state0 = env.base_env.get_safe_admission_state()
    initial_matrix, initial_totals = useful_gnb_totals()
    print("Scenario:", reset_info["scenario_name"])
    print(f"Action: b[g{source_gnb}->g{target_gnb},{slice_type}] = -1.0")
    print("Initial selected offset:", float(offsets[source_gnb, target_slot, slice_idx]), "dB")
    print("Initial live offset:", float(live_offsets[source_gnb, target_slot, slice_idx]), "dB")
    print("Initial source remaining quota:", state0["remaining"][(source_gnb, slice_type)])
    print("Initial direction quota:", state0["direction_quota"][(source_gnb, target_gnb, slice_type)])
    print("Initial useful load matrix rows g0/g1/g2, cols eMBB/URLLC/mMTC:")
    print(pd.DataFrame(initial_matrix, index=["g0", "g1", "g2"], columns=env.slice_types).round(3).to_string())
    print("Initial useful gNB totals:", np.round(initial_totals, 3).tolist())

    rows = []
    for settle_step in range(1, env.post_handover_settle_steps + 1):
        before_events = len(env.base_env.get_handover_events())
        before_offset = float(live_offsets[source_gnb, target_slot, slice_idx])
        before_matrix, before_totals = useful_gnb_totals()

        env.base_env.step(0)
        new_events = env.base_env.get_handover_events()[before_events:]
        updated_offsets = env._zero_quota_exhausted_offsets(live_offsets)
        zeroed_now = bool(
            updated_offsets[source_gnb, target_slot, slice_idx]
            != live_offsets[source_gnb, target_slot, slice_idx]
        )
        if not np.array_equal(updated_offsets, live_offsets):
            live_offsets = updated_offsets
            env._apply_slice_offsets(live_offsets)

        after_matrix, after_totals = useful_gnb_totals()
        state = env.base_env.get_safe_admission_state()
        row = {
            "settle_step": settle_step,
            "new_handovers": len(new_events),
            "routes": ", ".join(
                f"UE{event['ue_id']}:g{event['from_gnb']}->g{event['to_gnb']}"
                for event in new_events
            ) or "-",
            "source_used": state["used"].get((source_gnb, slice_type), 0),
            "source_remaining": state["remaining"].get((source_gnb, slice_type), 0),
            "direction_used": state["direction_used"].get((source_gnb, target_gnb, slice_type), 0),
            "direction_quota": state["direction_quota"].get((source_gnb, target_gnb, slice_type), 0),
            "offset_before_db": before_offset,
            "offset_after_db": float(live_offsets[source_gnb, target_slot, slice_idx]),
            "zeroed_now": zeroed_now,
        }
        row.update(load_fields("before", before_totals))
        row.update(load_fields("after", after_totals))
        row.update({
            f"g{idx}_delta": round(float(after_totals[idx] - before_totals[idx]), 3)
            for idx in range(env.n_gnbs)
        })
        rows.append(row)

    df = pd.DataFrame(rows)
    ordered_columns = [
        "settle_step", "new_handovers", "routes",
        "source_used", "source_remaining", "direction_used", "direction_quota",
        "offset_before_db", "offset_after_db", "zeroed_now",
        "g0_before", "g1_before", "g2_before",
        "g0_after", "g1_after", "g2_after",
        "g0_delta", "g1_delta", "g2_delta",
    ]
    print("\nSettle timeline with useful gNB load totals:")
    print(df[ordered_columns].to_string(index=False))
finally:
    env.close()

Scenario: high_load_inner_asymmetric
Action: b[g1->g2,eMBB] = -1.0
Initial selected offset: -6.0 dB
Initial live offset: -6.0 dB
Initial source remaining quota: 3
Initial direction quota: 3
Initial useful load matrix rows g0/g1/g2, cols eMBB/URLLC/mMTC:
    eMBB  URLLC  mMTC
g0  0.54    0.0   0.0
g1  0.90    0.0   0.0
g2  0.12    0.0   0.0
Initial useful gNB totals: [0.54, 0.9, 0.12]

Settle timeline with useful gNB load totals:
 settle_step  new_handovers     routes  source_used  source_remaining  direction_used  direction_quota  offset_before_db  offset_after_db  zeroed_now  g0_before  g1_before  g2_before  g0_after  g1_after  g2_after  g0_delta  g1_delta  g2_delta
           1              0          -            0                 3               0                3              -6.0             -6.0       False       0.54        0.9       0.12      0.58      1.00      0.14      0.04      0.10      0.02
           2              0          -            0                 3            

The key line is settle step `7`: the third handover consumes the direction quota (`direction_used = 3`, `direction_quota = 3`) and the shared source/slice quota (`source_remaining = 0`). At that exact point, the live offset changes from `-6.0 dB` to `0.0 dB`. The same row also shows the served useful-load move caused by that handover: `g1_delta` is negative and `g2_delta` is positive.
